In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from glob import glob
from matplotlib.ticker import MaxNLocator
from scipy.stats import skew, kurtosis
from statsmodels.stats.sandwich_covariance import cov_hac

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['axes.unicode_minus'] = False

In [2]:
fee = 0.001

indices = pd.read_csv("DB/Index_Version2_(CIZ)_CRSPdaily.csv",low_memory=False)

temp = indices[indices['IndNm'].isin(['CRSP Value-Weighted Index of the S&P 500 Universe'])][['DlyCalDt','COL1','IndNm']]
benchmark = temp.pivot(index = 'DlyCalDt',values='COL1',columns = 'IndNm')

benchmark.index.name = None
benchmark.columns.name = None

benchmark = benchmark[['CRSP Value-Weighted Index of the S&P 500 Universe']]
benchmark.columns = ['SP500']

benchmark = benchmark['2000-12-31'<benchmark.index].copy()
benchmark.loc['2001-01-01'] = 0
benchmark.index = pd.to_datetime(benchmark.index)
benchmark.sort_index(inplace=True)

temp = (1+benchmark).cumprod()

market = pd.read_csv("DB/market_portfolio.csv",index_col=0)[['EW_CAP100']]
market.index = pd.to_datetime(market.index)
temp['EW'] = np.exp(np.log(market).diff().fillna(0).cumsum())
idx_month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
idx_month_rt['EW'] = idx_month_rt['EW'] - 2 * fee

In [3]:
def cal_metric (rt):
    rt = rt.copy()
    annualized_return = f"{np.round(rt.mean()* 12,3):.3f}"
    annualized_vol = f"{np.round(rt.std() * np.sqrt(12),3):.3f}"
    # annualized_excess_return = f"{np.round((rt - r_f).mean()* 12,3):.3f}"
    donw_side_vol = f"{np.round(rt[rt<0].std() * np.sqrt(12),3):.3f}"
    sharpe_ratio = f"{np.round(float(annualized_return) / float(annualized_vol),3):.3f}"
    sortino_ratio = f"{np.round(float(annualized_return) / float(donw_side_vol),3):.3f}"
    # max_drawdown = f"{np.round((1 - (np.exp(np.rt.cumsum())/np.exp(rt.cumsum()).cummax()).min()) ,3):.3f}"
    cumulative_wealth = (1 + rt).cumprod()
    max_drawdown = f"{np.round(1 - (cumulative_wealth / cumulative_wealth.cummax()).min(),3):.3f}"
    calmar_ratio = f"{np.round(float(annualized_return) / float(max_drawdown),3):.3f}"
    return annualized_return,annualized_vol,donw_side_vol,sharpe_ratio,sortino_ratio, max_drawdown, calmar_ratio

In [4]:
models_seed_data = {} 
model_names = [
            # 'b16_30d_enc2',
            # 'b32_20d_enc2',
            # 'b32_25d_enc2',
            'b32_30d_enc2',
            # # 'b32_30d_enc4',
            # # 'b32_30d_enc6',
            'gray',
            'MOM',
            'STR']
seeds = ['avg'] # ['seed14', 'seed51', 'seed60', 'seed71', 'seed92'] 
mode = "equal_port" # equal_port , value_port

for name in model_names:
    seed_returns = []
    
    for seed in seeds:
        temp = pd.read_csv(f'backtest_vff/{name}/{seed}/cap_limit100/{mode}.csv', index_col=0)
        temp.columns = list(range(10,0,-1))
        temp.index = pd.to_datetime(temp.index)
        
        month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1) - 2 * fee

        result = month_rt.apply(cal_metric).astype(float)
        result.index = ["RET","VOL","DD","Sharpe","Sortino","MDD","Calmar"]
        seed_returns.append(result.loc['RET'])

    models_seed_data[name] = pd.DataFrame(seed_returns)

In [5]:
idx_result = idx_month_rt.apply(cal_metric).astype(float)
idx_result.index = ["RET","VOL","DD","Sharpe","Sortino","MDD","Calmar"]

In [6]:
idx_month_rt.apply(cal_metric).T

,0,1,2,3,4,5,6
SP500,0.079,0.153,0.113,0.516,0.699,0.485,0.163
EW,0.086,0.213,0.140,0.404,0.614,0.592,0.145


In [ ]:
plt.figure(figsize=(5, 5),dpi=400) # 

styles = [
    {"name": 'b32_30d_enc2', "color": "#0f4c81", "label": "ViT",  "zorder": 6}, # "linestyle": "-",
    {"name": 'gray', "color": "#DC343B", "label": "CNN",  "zorder": 5}, #"linestyle": "-.",
    {"name": 'MOM', "color": "#F9812A", "label": "MOM",  "zorder": 4}, # "linestyle": "--",
    {"name": 'STR', "color": "#006144", "label": "STR",  "zorder": 3}, # "linestyle": ":",
]

for style in styles:
    model_name = style["name"]
    data_df = models_seed_data[model_name] # (5 seeds x 10 deciles)
    plt.plot(data_df.columns, data_df.values[0],
        label=style["label"],
        color=style["color"],
        # linestyle=style["linestyle"],
        zorder = style["zorder"],
        marker = 'o',
        markersize=5)

plt.axhline(0.079,label ='S&P500',color='#3c3c3c', linewidth=2,zorder=2) # ,linestyle='-.'
plt.axhline(0.086,label ='EW',color='#888080', linewidth=2,zorder=1) # linestyle='--',

plt.xlabel("Signal Decile",fontsize=14)
plt.ylabel("Annualized Return",fontsize=14)
plt.xticks(data_df.columns,fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.grid(True, alpha=0.3)
# plt.show()
plt.tight_layout()
plt.savefig("fig_decile_prediction_performance_an_ret.png",dpi=400,transparent=True)
plt.close()

In [8]:
models_seed_data = {} 
model_names = [
            # 'b16_30d_enc2',
            # 'b32_20d_enc2',
            # 'b32_25d_enc2',
            'b32_30d_enc2',
            # # 'b32_30d_enc4',
            # # 'b32_30d_enc6',
            'gray',
            'MOM',
            'STR']
seeds = ['avg'] # ['seed14', 'seed51', 'seed60', 'seed71', 'seed92'] 
mode = "equal_port" # equal_port , value_port

for name in model_names:
    seed_returns = []
    
    for seed in seeds:
        temp = pd.read_csv(f'backtest_vff/{name}/{seed}/cap_limit100/{mode}.csv', index_col=0)
        temp.columns = list(range(10,0,-1))
        temp.index = pd.to_datetime(temp.index)
        
        month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1) - 2 * fee
        result = month_rt.apply(cal_metric).astype(float)
        result.index = ["RET","VOL","DD","Sharpe","Sortino","MDD","Calmar"]
        seed_returns.append(result.loc['VOL'])

    models_seed_data[name] = pd.DataFrame(seed_returns)

In [9]:
models_seed_data

{'b32_30d_enc2':         10     9      8      7      6     5      4      3      2      1 
 VOL  0.213  0.214  0.214  0.215  0.219  0.22  0.221  0.223  0.234  0.246,
 'gray':         10     9      8      7      6      5     4     3      2      1 
 VOL  0.205  0.213  0.215  0.216  0.224  0.223  0.23  0.23  0.233  0.224,
 'MOM':         10    9      8      7      6      5      4      3      2     1 
 VOL  0.222  0.18  0.164  0.165  0.175  0.187  0.213  0.244  0.301  0.42,
 'STR':         10     9      8      7      6     5      4      3      2     1 
 VOL  0.368  0.268  0.231  0.202  0.185  0.18  0.179  0.182  0.203  0.25}

In [ ]:
plt.figure(figsize=(5,5),dpi=400) # 

styles = [
    {"name": 'b32_30d_enc2', "color": "#0f4c81", "label": "ViT",  "zorder": 6}, # "linestyle": "-",
    {"name": 'gray', "color": "#DC343B", "label": "CNN",  "zorder": 5}, #"linestyle": "-.",
    {"name": 'MOM', "color": "#F9812A", "label": "MOM",  "zorder": 4}, # "linestyle": "--",
    {"name": 'STR', "color": "#006144", "label": "STR",  "zorder": 3}, # "linestyle": ":",
]

for style in styles:
    model_name = style["name"]
    data_df = models_seed_data[model_name] # (5 seeds x 10 deciles)
    plt.plot(data_df.columns, data_df.values[0],
        label=style["label"],
        color=style["color"],
        # linestyle=style["linestyle"],
        zorder = style["zorder"],
        marker = 'o',
        markersize=5)

plt.axhline(0.153,label ='S&P500',color='#3c3c3c', linewidth=2,zorder=2) # ,linestyle='-.'
plt.axhline(0.213,label ='EW',color='#888080', linewidth=2,zorder=1) # linestyle='--',

plt.xlabel("Signal Decile",fontsize=14)
plt.ylabel("Annualized Volatility",fontsize=14)
plt.xticks(data_df.columns,fontsize=14)
plt.yticks(fontsize=14)
# plt.legend(fontsize=14)
plt.grid(True, alpha=0.3)
# plt.show()
plt.tight_layout()
plt.ylim(0.11,0.44)
plt.savefig("fig_decile_prediction_performance_an_vol.png",dpi=400,transparent=True)
plt.close()

# Portfolio Performance

In [11]:
models_seed_data = {} 

month_return_table = []

seeds = ['avg']
mode = "equal_port" # equal_port , value_port

for name in ['b32_30d_enc2','gray']:
    seed_returns = []
    for seed in seeds:
        temp = pd.read_csv(f'backtest_vff/{name}/{seed}/cap_limit100/{mode}.csv', index_col=0)
        temp.columns = list(range(10,0,-1))
        temp.index = pd.to_datetime(temp.index)
        month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
        month_return_table.extend([month_rt[10] - 2*fee ,month_rt[1] - 2*fee , month_rt[10]-month_rt[1] - 2*fee])


for name in ['MOM','STR']:
    seed_returns = []
    for seed in seeds:
        temp = pd.read_csv(f'backtest_vff/{name}/{seed}/cap_limit100/{mode}.csv', index_col=0)
        temp.columns = list(range(10,0,-1))
        temp.index = pd.to_datetime(temp.index)
        month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
        month_return_table.append(month_rt[10]-month_rt[1] - 2*fee)

In [12]:
m_rt_df = pd.concat(month_return_table,axis=1)
m_rt_df.columns = ['ViT H','ViT L','ViT H-L',
                   'CNN H','CNN L','CNN H-L',
                   'MOM H-L', 'STR H-L',
                   ]

rt_df = pd.concat([m_rt_df,idx_month_rt],axis=1)

In [13]:
def nw_tstat(series):
    series = series.dropna()
    model = sm.OLS(series, np.ones(len(series)))
    result = model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    return result.tvalues.iloc[0] # ,result.pvalues.iloc[0]

In [14]:
rt_df.apply(nw_tstat)

ViT H      3.752968
ViT L     -0.408987
ViT H-L    3.546672
CNN H      3.498734
CNN L      0.162636
CNN H-L    2.677716
MOM H-L    0.730299
STR H-L    1.441416
SP500      2.777975
EW         1.973414
dtype: float64

In [15]:
summary_stat = rt_df.describe().T
summary_stat = summary_stat.drop(columns='count')
summary_stat['t-stat'] = rt_df.apply(nw_tstat)
summary_stat['Skew'] = rt_df.apply(lambda x: skew(x, bias=False))
summary_stat['Kurtosis'] = rt_df.apply(lambda x: kurtosis(x, fisher=False, bias=False))
summary_stat['Standard error'] = summary_stat['std'] / np.sqrt(rt_df.count())
summary_stat = summary_stat[['mean', 'std', 'Standard error', 
                              't-stat', 'min', '25%', '50%', 
                              '75%', 'max', 'Skew', 'Kurtosis']].T
summary_stat['blank'] = ''

In [16]:
summary_stat

,ViT H,ViT L,ViT H-L,CNN H,CNN L,CNN H-L,MOM H-L,STR H-L,SP500,EW,blank
mean,0.012008,-0.002067,0.012075,0.010685,0.000751,0.007934,0.004120,0.005493,0.006624,0.007150,
std,0.061420,0.070985,0.047876,0.059218,0.064670,0.045329,0.089509,0.069368,0.044184,0.061623,
Standard error,0.003546,0.004098,0.002764,0.003419,0.003734,0.002617,0.005168,0.004005,0.002551,0.003558,
t-stat,3.752968,-0.408987,3.546672,3.498734,0.162636,2.677716,0.730299,1.441416,2.777975,1.973414,
min,-0.267828,-0.228788,-0.317714,-0.280080,-0.223450,-0.321973,-0.731728,-0.168104,-0.163363,-0.244054,
25%,-0.018339,-0.043178,-0.006312,-0.017992,-0.035383,-0.009361,-0.024072,-0.027663,-0.015643,-0.026679,
50%,0.011753,-0.005046,0.014002,0.013350,0.000811,0.010752,0.013427,-0.004190,0.010946,0.006517,
75%,0.041518,0.032745,0.033606,0.036558,0.034352,0.029209,0.051297,0.026112,0.033068,0.041360,
max,0.459326,0.351262,0.309814,0.351485,0.355848,0.220802,0.205820,0.603204,0.180167,0.266158,
Skew,1.322416,0.589447,-0.254839,0.481111,0.641051,-1.009552,-2.673834,3.192253,-0.351113,0.207440,


In [17]:
summary_stat = pd.concat([summary_stat[['blank']],
           summary_stat[['ViT H', 'ViT L', 'ViT H-L']],
           summary_stat[['blank']],
           summary_stat[['CNN H', 'CNN L', 'CNN H-L']],
           summary_stat[['blank']],
           summary_stat[['MOM H-L', 'STR H-L']],
           summary_stat[['blank']],
           summary_stat[['SP500', 'EW']],
           ],axis=1)

In [18]:
print(np.round(summary_stat,3).to_latex(float_format="%.3f" ))

\begin{tabular}{llrrrlrrrlrrlrr}
\toprule
 & blank & ViT H & ViT L & ViT H-L & blank & CNN H & CNN L & CNN H-L & blank & MOM H-L & STR H-L & blank & SP500 & EW \\
\midrule
mean &  & 0.012 & -0.002 & 0.012 &  & 0.011 & 0.001 & 0.008 &  & 0.004 & 0.005 &  & 0.007 & 0.007 \\
std &  & 0.061 & 0.071 & 0.048 &  & 0.059 & 0.065 & 0.045 &  & 0.090 & 0.069 &  & 0.044 & 0.062 \\
Standard error &  & 0.004 & 0.004 & 0.003 &  & 0.003 & 0.004 & 0.003 &  & 0.005 & 0.004 &  & 0.003 & 0.004 \\
t-stat &  & 3.753 & -0.409 & 3.547 &  & 3.499 & 0.163 & 2.678 &  & 0.730 & 1.441 &  & 2.778 & 1.973 \\
min &  & -0.268 & -0.229 & -0.318 &  & -0.280 & -0.223 & -0.322 &  & -0.732 & -0.168 &  & -0.163 & -0.244 \\
25% &  & -0.018 & -0.043 & -0.006 &  & -0.018 & -0.035 & -0.009 &  & -0.024 & -0.028 &  & -0.016 & -0.027 \\
50% &  & 0.012 & -0.005 & 0.014 &  & 0.013 & 0.001 & 0.011 &  & 0.013 & -0.004 &  & 0.011 & 0.007 \\
75% &  & 0.042 & 0.033 & 0.034 &  & 0.037 & 0.034 & 0.029 &  & 0.051 & 0.026 &  & 0.033 & 0.041 

In [19]:
summary_risk_met = rt_df.apply(cal_metric) 
summary_risk_met.index = ['Annualized return', 'Annualized volatility', 'Downside deviation', 'Sharpe ratio', 'Sortino ratio', 'Maximum drawdown', 'Calmar ratio']

In [20]:
gross_profit = rt_df[(rt_df > 0)].sum()
gross_loss = rt_df[(rt_df < 0)].sum()
profit_factor = (gross_profit / abs(gross_loss))


yearly_returns = rt_df.groupby(rt_df.index.year).sum()

# Count profitable vs unprofitable years
profitable_years = (yearly_returns > 0).sum()
unprofitable_years = (yearly_returns <= 0).sum()

In [21]:
rt_df[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].to_csv('DB/monthly_rt.csv')

In [22]:

risk_metric = summary_risk_met.T
risk_metric['Gross profit'] = np.round(gross_profit,3)
risk_metric['Gross loss'] = np.round(gross_loss,3)
risk_metric['Profit factor'] = np.round(profit_factor,3) 
risk_metric['Profitable years'] = profitable_years
risk_metric['Unprofitable years'] = unprofitable_years

risk_metric = risk_metric.T

In [23]:
risk_metric['blank'] = ''

In [24]:
risk_metric = pd.concat([risk_metric[['blank']],
           risk_metric[['ViT H', 'ViT L', 'ViT H-L']],
           risk_metric[['blank']],
           risk_metric[['CNN H', 'CNN L', 'CNN H-L']],
           risk_metric[['blank']],
           risk_metric[['MOM H-L', 'STR H-L']],
           risk_metric[['blank']],
           risk_metric[['SP500', 'EW']],
           ],axis=1)

In [25]:
risk_metric

,blank,ViT H,ViT L,ViT H-L,blank,CNN H,CNN L,CNN H-L,blank,MOM H-L,STR H-L,blank,SP500,EW
Annualized return,,0.144,-0.025,0.145,,0.128,0.009,0.095,,0.049,0.066,,0.079,0.086
Annualized volatility,,0.213,0.246,0.166,,0.205,0.224,0.157,,0.310,0.240,,0.153,0.213
Downside deviation,,0.131,0.150,0.143,,0.140,0.138,0.141,,0.325,0.105,,0.113,0.140
Sharpe ratio,,0.676,-0.102,0.873,,0.624,0.040,0.605,,0.158,0.275,,0.516,0.404
Sortino ratio,,1.099,-0.167,1.014,,0.914,0.065,0.674,,0.151,0.629,,0.699,0.614
Maximum drawdown,,0.481,0.872,0.617,,0.479,0.757,0.591,,0.841,0.580,,0.485,0.592
Calmar ratio,,0.299,-0.029,0.235,,0.267,0.012,0.161,,0.058,0.114,,0.163,0.145
Gross profit,,8.204,7.464,6.844,,7.947,7.104,5.862,,9.435,7.168,,6.073,7.852
Gross loss,,-4.602,-8.084,-3.222,,-4.742,-6.879,-3.482,,-8.199,-5.52,,-4.086,-5.707
Profit factor,,1.783,0.923,2.124,,1.676,1.033,1.684,,1.151,1.299,,1.486,1.376


In [26]:
print(risk_metric.to_latex(float_format="%.3f" ))

\begin{tabular}{lllllllllllllll}
\toprule
 & blank & ViT H & ViT L & ViT H-L & blank & CNN H & CNN L & CNN H-L & blank & MOM H-L & STR H-L & blank & SP500 & EW \\
\midrule
Annualized return &  & 0.144 & -0.025 & 0.145 &  & 0.128 & 0.009 & 0.095 &  & 0.049 & 0.066 &  & 0.079 & 0.086 \\
Annualized volatility &  & 0.213 & 0.246 & 0.166 &  & 0.205 & 0.224 & 0.157 &  & 0.310 & 0.240 &  & 0.153 & 0.213 \\
Downside deviation &  & 0.131 & 0.150 & 0.143 &  & 0.140 & 0.138 & 0.141 &  & 0.325 & 0.105 &  & 0.113 & 0.140 \\
Sharpe ratio &  & 0.676 & -0.102 & 0.873 &  & 0.624 & 0.040 & 0.605 &  & 0.158 & 0.275 &  & 0.516 & 0.404 \\
Sortino ratio &  & 1.099 & -0.167 & 1.014 &  & 0.914 & 0.065 & 0.674 &  & 0.151 & 0.629 &  & 0.699 & 0.614 \\
Maximum drawdown &  & 0.481 & 0.872 & 0.617 &  & 0.479 & 0.757 & 0.591 &  & 0.841 & 0.580 &  & 0.485 & 0.592 \\
Calmar ratio &  & 0.299 & -0.029 & 0.235 &  & 0.267 & 0.012 & 0.161 &  & 0.058 & 0.114 &  & 0.163 & 0.145 \\
Gross profit &  & 8.204 & 7.464 & 6.844 &  

In [27]:
cumrt_df = rt_df.copy()
cumrt_df.loc[pd.to_datetime('2001-01-01')] = 0
cumrt_df.sort_index(inplace=True)

In [28]:
plt.figure(figsize=(6*1.618,6),dpi=400) # 

plt.plot(cumrt_df.index, np.log(cumrt_df['ViT H-L']+1).cumsum(),color= "#0f4c81", label="ViT", linewidth=2 , zorder= 6)
plt.plot(cumrt_df.index, np.log(cumrt_df['CNN H-L']+1).cumsum(),color= "#DC343B", label="CNN", linewidth=2 , zorder= 5)
plt.plot(cumrt_df.index, np.log(cumrt_df['MOM H-L']+1).cumsum(),color= "#F9812A", label="MOM", linewidth=2 , zorder= 4)
plt.plot(cumrt_df.index, np.log(cumrt_df['STR H-L']+1).cumsum(),color= "#006144", label="STR", linewidth=2 , zorder= 3)
plt.plot(cumrt_df.index, np.log(cumrt_df['SP500']+1).cumsum(),color= "#3c3c3c", label="S&P500", linewidth=2 , zorder= 2)
plt.plot(cumrt_df.index, np.log(cumrt_df['EW']+1).cumsum(),color= "#888080", label="EW", linewidth=2 , zorder= 1)

plt.legend(fontsize=14)
plt.yticks(fontsize=14)
plt.xticks(fontsize=14)
plt.grid()

plt.tight_layout()
# plt.ylim(0.11,0.44)
plt.savefig("fig_cum_port.png",dpi=400,transparent=True)
plt.close()


# sub period

In [ ]:
# (a)2001–2006
# (b)2007–2009
# (c)2010–2019
# (d)2020–2020
# (e)2021-2025

In [30]:
sub1 = rt_df[rt_df.index<='2007-01-01'].apply(cal_metric)
sub2 = rt_df[('2007-01-01'<=rt_df.index) & (rt_df.index<='2009-12-31')].apply(cal_metric)
sub3 = rt_df[('2010-01-01'<=rt_df.index) & (rt_df.index<='2019-12-31')].apply(cal_metric)
sub4 = rt_df[('2020-01-01'<=rt_df.index) & (rt_df.index<='2020-12-31')].apply(cal_metric)
sub5 = rt_df[('2021-01-01'<=rt_df.index) & (rt_df.index<='2025-12-31')].apply(cal_metric)

for sub in [sub1,sub2,sub3,sub4,sub5]:
    sub.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]

In [31]:
temp = pd.concat([
    sub1[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].loc[['Annualized return','Sharpe ratio','Maximum drawdown']],
    sub2[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].loc[['Annualized return','Sharpe ratio','Maximum drawdown']],
    sub3[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].loc[['Annualized return','Sharpe ratio','Maximum drawdown']],
    sub4[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].loc[['Annualized return','Sharpe ratio','Maximum drawdown']],
    sub5[['ViT H-L','CNN H-L','MOM H-L','STR H-L','SP500','EW']].loc[['Annualized return','Sharpe ratio','Maximum drawdown']],
],axis=0)

In [32]:
temp['blank'] = ''

In [33]:
print(pd.concat([temp['blank'],temp.iloc[:,:-1]],axis=1).to_latex(float_format="%.3f" ))

\begin{tabular}{llllllll}
\toprule
 & blank & ViT H-L & CNN H-L & MOM H-L & STR H-L & SP500 & EW \\
\midrule
Annualized return &  & 0.133 & 0.051 & -0.097 & 0.212 & 0.003 & 0.162 \\
Sharpe ratio &  & 0.801 & 0.402 & -0.224 & 0.662 & 0.024 & 0.798 \\
Maximum drawdown &  & 0.343 & 0.359 & 0.663 & 0.214 & 0.388 & 0.212 \\
Annualized return &  & 0.205 & 0.213 & -0.190 & 0.131 & -0.026 & -0.015 \\
Sharpe ratio &  & 1.145 & 1.139 & -0.511 & 0.483 & -0.123 & -0.053 \\
Maximum drawdown &  & 0.211 & 0.121 & 0.771 & 0.226 & 0.485 & 0.592 \\
Annualized return &  & 0.118 & 0.076 & 0.118 & 0.023 & 0.119 & 0.085 \\
Sharpe ratio &  & 1.146 & 0.731 & 0.656 & 0.180 & 0.952 & 0.518 \\
Maximum drawdown &  & 0.120 & 0.139 & 0.263 & 0.283 & 0.157 & 0.256 \\
Annualized return &  & -0.502 & -0.475 & -0.676 & 0.511 & 0.178 & 0.416 \\
Sharpe ratio &  & -2.684 & -2.654 & -2.364 & 1.118 & 0.569 & 0.924 \\
Maximum drawdown &  & 0.390 & 0.368 & 0.521 & 0.197 & 0.235 & 0.303 \\
Annualized return &  & 0.307 & 0.230 

# Factor Regression

In [34]:
ff3 = pd.read_csv("DB/F-F_Research_Data_Factors.csv",index_col=0).drop(columns = ['RF'])
ff5 = pd.read_csv("DB/F-F_Research_Data_5_Factors_2x3.csv",index_col=0).drop(columns = ['RF'])
ffm = pd.read_csv("DB/F-F_Momentum_Factor.csv",index_col=0)
ffs = pd.read_csv("DB/F-F_ST_Reversal_Factor.csv",index_col=0)
q5 = pd.read_csv("DB/q5_factors_monthly_2024.csv",index_col=0).drop(columns = ['R_F'])

ff3 = ff3[(200101<=ff3.index) & (ff3.index<=202512)].copy()
ff5 = ff5[(200101<=ff5.index) & (ff5.index<=202512)].copy()
ffm = ffm[(200101<=ffm.index) & (ffm.index<=202512)].copy()
ffs = ffs[(200101<=ffs.index) & (ffs.index<=202512)].copy()
q5 = q5[(2001<=q5.index) & (q5.index <=2024)].drop(columns = ['month']).copy()

ff3.columns = ['Market','SMB', 'HML']
ffm.columns = ['Momentum']
ffs.columns = ['Reversal']
ff5.columns = ['Market','SMB5','HML5','RMW5','CMA5']
q5.columns = ['Market','R_ME', 'R_IA', 'R_ROE', 'R_EG']

In [35]:
def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

In [36]:
total_table_factor_reg = []

for i in [0,1]:
    mdl = ['ViT','CNN'][i]

    factor_reg_result = []
    r2_result = []

    for factor_port in [ff3, pd.concat([ff3,ffm,ffs],axis=1),ff5,q5]:
        X = factor_port
        X = sm.add_constant(X)
        y = ((rt_df[mdl + ' H-L'] + 2* fee )* 100).copy()
        y = y.iloc[:len(X)]
        model = sm.OLS(y.values, X.values)
        results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})

        coef_df = pd.DataFrame(columns = list(X.columns))
        coef_df.loc['coef'] = list(np.array([f"{v:.3f}" for v in results.params]) + np.array(list(map(get_stars,results.pvalues))))

        coef_and_tstat = []
        for idx in range(len(coef_df.columns)):
            col = coef_df.columns[idx]
            coef_and_tstat.append(coef_df[[col]])

            temp = coef_df[[col]].copy()
            col = col + str('_tstat')
            temp[col] = [f"({v:.3f})" for v in results.tvalues][idx]
            coef_and_tstat.append(temp[[col]])
        
        factor_reg_df = pd.concat(coef_and_tstat,axis=1)
        factor_reg_result.append(factor_reg_df.T)
        r2_result.append([f"{results.rsquared:.3f}",f"{results.rsquared_adj:.3f}",str(len(X))])

    model_temp = pd.concat(factor_reg_result,axis=1)
    model_temp.columns = ['FF3','FF3+2','FF5','$q^5$']

    r2_result_df = pd.DataFrame(r2_result).T
    r2_result_df.index = ['R2','Adj. R2','Num. obs.']
    r2_result_df.columns = ['FF3','FF3+2','FF5','$q^5$']

    model_temp = pd.concat([model_temp,r2_result_df]).fillna(' ')
    model_temp['blank'] = ''
    total_table_factor_reg.append(pd.concat([model_temp['blank'],model_temp.iloc[:,:4]],axis=1))

factor_reg_df =  pd.concat(total_table_factor_reg,axis=1)

In [37]:
factor_reg_df.index = ['const','','Market','','SMB','','HML','','Momentum','','Reversal','','SMB5','','HML5','','RMW5','','CMA','','R\_ME','','R\_IA','','R\_ROE','','R\_EG','','R2','Adj. R2','Num. obs.']

In [38]:
print(factor_reg_df.to_latex())

\begin{tabular}{lllllllllll}
\toprule
 & blank & FF3 & FF3+2 & FF5 & $q^5$ & blank & FF3 & FF3+2 & FF5 & $q^5$ \\
\midrule
const &  & 1.432*** & 1.441*** & 1.227*** & 1.212*** &  & 0.916*** & 0.936*** & 0.793*** & 0.657* \\
 &  & (4.631) & (4.471) & (3.690) & (2.890) &  & (3.273) & (3.285) & (2.685) & (1.825) \\
Market &  & 0.018 & -0.044 & 0.060 & 0.120 &  & 0.166*** & 0.112* & 0.184*** & 0.254*** \\
 &  & (0.340) & (-0.472) & (1.079) & (1.131) &  & (3.373) & (1.770) & (3.772) & (3.226) \\
SMB &  & -0.403* & -0.451** &   &   &  & -0.388** & -0.424** &   &   \\
 &  & (-1.940) & (-2.385) &   &   &  & (-2.020) & (-2.294) &   &   \\
HML &  & 0.134 & 0.115 &   &   &  & 0.106 & 0.088 &   &   \\
 &  & (1.291) & (0.952) &   &   &  & (1.013) & (0.781) &   &   \\
Momentum &  &   & -0.039 &   &   &  &   & -0.058 &   &   \\
 &  &   & (-0.291) &   &   &  &   & (-0.587) &   &   \\
Reversal &  &   & 0.197 &   &   &  &   & 0.124 &   &   \\
 &  &   & (1.644) &   &   &  &   & (1.460) &   &   \\
SMB5 & 

# Size and Cost

In [39]:
glob("backtest_vff/b32_30d_enc2/*/*/equal_port.csv")

['backtest_vff/b32_30d_enc2\\avg\\cap_limit100\\equal_port.csv',
 'backtest_vff/b32_30d_enc2\\avg\\cap_limit50\\equal_port.csv',
 'backtest_vff/b32_30d_enc2\\avg\\cap_limit60\\equal_port.csv',
 'backtest_vff/b32_30d_enc2\\avg\\cap_limit70\\equal_port.csv',
 'backtest_vff/b32_30d_enc2\\avg\\cap_limit80\\equal_port.csv',
 'backtest_vff/b32_30d_enc2\\avg\\cap_limit90\\equal_port.csv']

In [40]:
cap_sensitivity = []
for cap_limit_level in [100,90,80,70,60,50]:
    cap_month_return_table = []
    for model in ['b32_30d_enc2','gray','MOM','STR']:

            csv_path = f"backtest_vff/{model}/avg/cap_limit{cap_limit_level}/equal_port.csv"
            temp = pd.read_csv(csv_path,index_col=0)
            temp.columns = list(range(10,0,-1))
            temp.index = pd.to_datetime(temp.index)
            month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
            cap_month_return_table.extend([month_rt[10]-month_rt[1] - 2 * fee])
    mdl_cap_sum_table = pd.concat(cap_month_return_table,axis=1).apply(cal_metric)
    mdl_cap_sum_table.columns = ['ViT','CNN','MOM','STR']# ['100','90','80','70','60','50']
    mdl_cap_sum_table.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]

    cap_sensitivity.append(mdl_cap_sum_table.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']])

cap_sensitivity = pd.concat(cap_sensitivity)

In [41]:
vw_cap_sensitivity = []
for cap_limit_level in [100,90,80,70,60,50]:
    cap_month_return_table = []
    for model in ['b32_30d_enc2','gray','MOM','STR']:

            csv_path = f"backtest_vff/{model}/avg/cap_limit{cap_limit_level}/value_port.csv"
            temp = pd.read_csv(csv_path,index_col=0)
            temp.columns = list(range(10,0,-1))
            temp.index = pd.to_datetime(temp.index)
            month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
            cap_month_return_table.extend([month_rt[10]-month_rt[1] - 2 * fee])
    mdl_cap_sum_table = pd.concat(cap_month_return_table,axis=1).apply(cal_metric)
    mdl_cap_sum_table.columns = ['ViT','CNN','MOM','STR']# ['100','90','80','70','60','50']
    mdl_cap_sum_table.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]

    vw_cap_sensitivity.append(mdl_cap_sum_table.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']])

vw_cap_sensitivity = pd.concat(vw_cap_sensitivity)

In [42]:
cap_sensitivity

,ViT,CNN,MOM,STR
Annualized return,0.145,0.095,0.049,0.066
Sharpe ratio,0.873,0.605,0.158,0.275
Maximum drawdown,0.617,0.591,0.841,0.580
Annualized return,0.126,0.095,0.044,0.016
Sharpe ratio,1.024,0.833,0.154,0.073
Maximum drawdown,0.405,0.379,0.768,0.572
Annualized return,0.091,0.063,0.030,0.012
Sharpe ratio,0.820,0.624,0.107,0.056
Maximum drawdown,0.388,0.371,0.783,0.576
Annualized return,0.070,0.038,0.014,0.018


In [43]:
cap_sensitivity['blank'] = ''

In [44]:
ew_cap = pd.read_csv("DB/market_portfolio.csv",index_col=0)
ew_cap.index = pd.to_datetime(ew_cap.index)
ew_cap = np.exp(np.log(ew_cap).diff().fillna(0).cumsum())
ew_cap = ((ew_cap.resample('ME').last() / ew_cap.resample('ME').first()) - 1) - 2 * fee
ew_cap = ew_cap.apply(cal_metric)
ew_cap.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]
ew_cap = ew_cap.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']]

In [45]:
cap_sensitivity = pd.concat([cap_sensitivity['blank'], cap_sensitivity.iloc[:,:-1],cap_sensitivity['blank'],cap_sensitivity['blank']],axis=1)

In [46]:
cap_sensitivity.columns = ['', 'ViT', 'CNN', 'MOM', 'STR', 'SP500', 'EW']

In [47]:
cap_sensitivity['EW'] = pd.concat([ew_cap[col] for col in ew_cap.columns])

In [48]:
temp = idx_month_rt.apply(cal_metric)[['SP500']]
temp.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]
temp = temp.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']]

In [49]:
cap_sensitivity['SP500'] = pd.concat([temp for i in range(6)],axis=0)

In [50]:
print(pd.concat([cap_sensitivity[['', 'ViT', 'CNN', 'MOM', 'STR']],cap_sensitivity[['']],vw_cap_sensitivity,cap_sensitivity[['']],cap_sensitivity[['SP500','EW']]],axis=1).to_latex())

\begin{tabular}{llllllllllllll}
\toprule
 &  & ViT & CNN & MOM & STR &  & ViT & CNN & MOM & STR &  & SP500 & EW \\
\midrule
Annualized return &  & 0.145 & 0.095 & 0.049 & 0.066 &  & 0.066 & 0.017 & 0.063 & 0.022 &  & 0.079 & 0.086 \\
Sharpe ratio &  & 0.873 & 0.605 & 0.158 & 0.275 &  & 0.434 & 0.120 & 0.168 & 0.079 &  & 0.516 & 0.404 \\
Maximum drawdown &  & 0.617 & 0.591 & 0.841 & 0.580 &  & 0.535 & 0.500 & 0.884 & 0.675 &  & 0.485 & 0.592 \\
Annualized return &  & 0.126 & 0.095 & 0.044 & 0.016 &  & 0.043 & 0.025 & 0.053 & 0.004 &  & 0.079 & 0.082 \\
Sharpe ratio &  & 1.024 & 0.833 & 0.154 & 0.073 &  & 0.336 & 0.205 & 0.158 & 0.015 &  & 0.516 & 0.392 \\
Maximum drawdown &  & 0.405 & 0.379 & 0.768 & 0.572 &  & 0.516 & 0.508 & 0.868 & 0.762 &  & 0.485 & 0.589 \\
Annualized return &  & 0.091 & 0.063 & 0.030 & 0.012 &  & 0.064 & 0.014 & 0.046 & 0.006 &  & 0.079 & 0.085 \\
Sharpe ratio &  & 0.820 & 0.624 & 0.107 & 0.056 &  & 0.552 & 0.125 & 0.142 & 0.024 &  & 0.516 & 0.405 \\
Maximum drawd

In [51]:
print(cap_sensitivity.to_latex())

\begin{tabular}{llllllll}
\toprule
 &  & ViT & CNN & MOM & STR & SP500 & EW \\
\midrule
Annualized return &  & 0.145 & 0.095 & 0.049 & 0.066 & 0.079 & 0.086 \\
Sharpe ratio &  & 0.873 & 0.605 & 0.158 & 0.275 & 0.516 & 0.404 \\
Maximum drawdown &  & 0.617 & 0.591 & 0.841 & 0.580 & 0.485 & 0.592 \\
Annualized return &  & 0.126 & 0.095 & 0.044 & 0.016 & 0.079 & 0.082 \\
Sharpe ratio &  & 1.024 & 0.833 & 0.154 & 0.073 & 0.516 & 0.392 \\
Maximum drawdown &  & 0.405 & 0.379 & 0.768 & 0.572 & 0.485 & 0.589 \\
Annualized return &  & 0.091 & 0.063 & 0.030 & 0.012 & 0.079 & 0.085 \\
Sharpe ratio &  & 0.820 & 0.624 & 0.107 & 0.056 & 0.516 & 0.405 \\
Maximum drawdown &  & 0.388 & 0.371 & 0.783 & 0.576 & 0.485 & 0.580 \\
Annualized return &  & 0.070 & 0.038 & 0.014 & 0.018 & 0.079 & 0.084 \\
Sharpe ratio &  & 0.700 & 0.404 & 0.052 & 0.085 & 0.516 & 0.400 \\
Maximum drawdown &  & 0.318 & 0.334 & 0.821 & 0.498 & 0.485 & 0.569 \\
Annualized return &  & 0.058 & 0.040 & 0.018 & 0.017 & 0.079 & 0.080 \\


# Cost

In [52]:
cost_sensitivity = []

for fee in [0.001,0.002,0.003,0.004,0.005]:
    cost_month_return_table = []
    for model in ['b32_30d_enc2','gray','MOM','STR']:

        csv_path = f"backtest_vff/{model}/avg/cap_limit100/equal_port.csv"
        temp = pd.read_csv(csv_path,index_col=0)
        temp.columns = list(range(10,0,-1))
        temp.index = pd.to_datetime(temp.index)
        month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
        cost_month_return_table.extend([month_rt[10]-month_rt[1] - 2*fee])

    mdl_cost_sum_table = pd.concat(cost_month_return_table,axis=1).apply(cal_metric)
    mdl_cost_sum_table.columns = ['ViT','CNN','MOM','STR']# ['100','90','80','70','60','50']
    mdl_cost_sum_table.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]

    cost_sensitivity.append(mdl_cost_sum_table.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']])

cost_sensitivity = pd.concat(cost_sensitivity)

In [53]:
temp = idx_month_rt.apply(cal_metric)[['SP500']]
temp.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]
temp = temp.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']]
cost_sensitivity['SP500'] = pd.concat([temp for i in range(5)],axis=0)

In [54]:
ew_cap = pd.read_csv("DB/market_portfolio.csv",index_col=0)[['EW_CAP100']]
ew_cap.columns = ["EW"]
ew_cap.index = pd.to_datetime(ew_cap.index)
ew_cap = np.exp(np.log(ew_cap).diff().fillna(0).cumsum())
ew_cap = ((ew_cap.resample('ME').last() / ew_cap.resample('ME').first()) - 1)

ew_fee_lst = []

for fee in [0.001,0.002,0.003,0.004,0.005]:
    temp = (ew_cap - 2*fee).apply(cal_metric)
    temp.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]
    temp = temp.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']]
    ew_fee_lst.append(temp)

cost_sensitivity['EW'] = pd.concat(ew_fee_lst,axis=0)

In [55]:
cost_sensitivity['blank'] = ''

In [56]:
print(pd.concat([cost_sensitivity['blank'],cost_sensitivity.iloc[:,:-1]],axis=1).to_latex(float_format="%.3f" ))

\begin{tabular}{llllllll}
\toprule
 & blank & ViT & CNN & MOM & STR & SP500 & EW \\
\midrule
Annualized return &  & 0.145 & 0.095 & 0.049 & 0.066 & 0.079 & 0.086 \\
Sharpe ratio &  & 0.873 & 0.605 & 0.158 & 0.275 & 0.516 & 0.404 \\
Maximum drawdown &  & 0.617 & 0.591 & 0.841 & 0.580 & 0.485 & 0.592 \\
Annualized return &  & 0.121 & 0.071 & 0.025 & 0.042 & 0.079 & 0.062 \\
Sharpe ratio &  & 0.729 & 0.452 & 0.081 & 0.175 & 0.516 & 0.291 \\
Maximum drawdown &  & 0.629 & 0.605 & 0.894 & 0.629 & 0.485 & 0.610 \\
Annualized return &  & 0.097 & 0.047 & 0.001 & 0.018 & 0.079 & 0.038 \\
Sharpe ratio &  & 0.584 & 0.299 & 0.003 & 0.075 & 0.516 & 0.178 \\
Maximum drawdown &  & 0.641 & 0.618 & 0.934 & 0.712 & 0.485 & 0.627 \\
Annualized return &  & 0.073 & 0.023 & -0.023 & -0.006 & 0.079 & 0.014 \\
Sharpe ratio &  & 0.440 & 0.146 & -0.074 & -0.025 & 0.516 & 0.066 \\
Maximum drawdown &  & 0.652 & 0.666 & 0.959 & 0.801 & 0.485 & 0.643 \\
Annualized return &  & 0.049 & -0.001 & -0.047 & -0.030 & 0.079

# Model Sens

In [57]:
model_sensitivity = []
fee = 0.001

model_month_return_table = []
path_lst = [f"backtest_vff/b32_30d_enc2/avg/cap_limit100/equal_port.csv",
            f"backtest_vff/b32_20d_enc2/avg/cap_limit100/equal_port.csv",
            f"backtest_vff/b32_25d_enc2/avg/cap_limit100/equal_port.csv",
            f"backtest_vff/b32_30d_enc4/avg/cap_limit100/equal_port.csv",
            f"backtest_vff/b32_30d_enc6/avg/cap_limit100/equal_port.csv",
            f"backtest_vff/b16_30d_enc2/avg/cap_limit100/equal_port.csv",
            ]

for csv_path in path_lst:
    temp = pd.read_csv(csv_path,index_col=0)
    temp.columns = list(range(10,0,-1))
    temp.index = pd.to_datetime(temp.index)
    month_rt = ((temp.resample('ME').last() / temp.resample('ME').first()) - 1)
    model_month_return_table.extend([month_rt[10]-month_rt[1] - 2*fee])

mdl_model_sum_table = pd.concat(model_month_return_table,axis=1).apply(cal_metric)
mdl_model_sum_table.columns = ['Baseline (D=20, L=2)','D=20','D=25','L=4','L=6','P=32']# ['100','90','80','70','60','50']
mdl_model_sum_table.index = ["Annualized return","VOL","DD","Sharpe ratio","Sortino","Maximum drawdown","Calmar"]

model_sensitivity.append(mdl_model_sum_table.loc[["Annualized return",'Sharpe ratio','Maximum drawdown']])

model_sensitivity = pd.concat(model_sensitivity)

In [58]:
model_sensitivity['blank'] = ''

In [59]:
print(pd.concat([model_sensitivity['blank'],model_sensitivity.iloc[:,:-1]],axis=1).to_latex())

\begin{tabular}{llllllll}
\toprule
 & blank & Baseline (D=20, L=2) & D=20 & D=25 & L=4 & L=6 & P=32 \\
\midrule
Annualized return &  & 0.145 & 0.138 & 0.126 & 0.153 & 0.150 & 0.168 \\
Sharpe ratio &  & 0.873 & 0.862 & 0.759 & 0.895 & 0.882 & 1.012 \\
Maximum drawdown &  & 0.617 & 0.610 & 0.672 & 0.609 & 0.623 & 0.575 \\
\bottomrule
\end{tabular}

